# 07 · Power BI export
Exports the star schema to `power_bi/data/` and generates the Power BI project (`power_bi/pbip/FarmerRevenueIntelligence.pbip`): the TMDL model with relationships and 62 DAX measures, plus six PBIR report pages.

Open the `.pbip` in Power BI Desktop (Options → Preview features: Power BI Project, TMDL and PBIR enabled), then click **Refresh**.

| | |
|---|---|
| **Objective** | Export the model and build the Power BI project. |
| **Dataset** | The SQLite database. |
| **Business question** | Is the data ready for an interactive report? |
| **Method** | CSV export, PBIP generation (TMDL model + PBIR report). |
| **Output** | power_bi/data/*.csv and power_bi/pbip/ (8 pages, 81 DAX measures). |
| **Key finding** | The report is regenerated from code, so it always matches the data. |

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import RAW, CURATED, PROC, RESULTS, FIGURES, DB_PATH, connect_ro
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
BLUE, ORANGE, AQUA, YELLOW, PINK, RED, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#e34948", "#9aa19c"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "axes.titleweight": "bold", "axes.titlesize": 11})
def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", bbox_inches="tight"); plt.show()
from src import export_powerbi
export_powerbi.main()

dim_crop                           26
dim_state                          36
dim_location                      760
dim_year                           37
dim_market                      3,418
dim_date                           67


fact_production               239,760


fact_district_crop_year       199,125


fact_state_crop_year           11,479
fact_msp                          516


fact_market_price             106,885
fact_cost_of_cultivation          300
fact_weather_state                910
fact_weather_subdivision        4,116
dq_issues                          18
dq_coverage                        67
dq_validation                       8
ref_sources                         9


In [2]:
import runpy
runpy.run_path(str(ROOT / "power_bi" / "build_pbip.py"), run_name="__main__")
pb = ROOT / "power_bi" / "pbip"
pages = sorted(p.name for p in (pb / "FarmerRevenueIntelligence.Report" / "definition" / "pages").iterdir() if p.is_dir())
visuals = {p: len(list((pb / "FarmerRevenueIntelligence.Report" / "definition" / "pages" / p / "visuals").iterdir())) for p in pages}
measures = (pb / "FarmerRevenueIntelligence.SemanticModel" / "definition" / "tables" / "_Measures.tmdl").read_text().count("\tmeasure ")
print("pages:", visuals); print("DAX measures:", measures)

written /home/claude/ifri/power_bi/pbip
pages: {'p1_overview': 17, 'p2_crops': 9, 'p3_states': 8, 'p4_mandi': 14, 'p5_weather': 12, 'p6_quality': 11, 'p7_season2026': 13, 'p8_explorer': 18}
DAX measures: 81


In [3]:
pd.DataFrame([{"file": f.name, "rows": sum(1 for _ in open(f, encoding="utf-8")) - 1, "MB": round(f.stat().st_size / 1e6, 2)} for f in sorted((ROOT / "power_bi" / "data").glob("*.csv"))])

,file,rows,MB
0,agg_q04b_value_growth_decomposition.csv,22,0.00
1,agg_q05a_cross_market_dispersion_2024.csv,21,0.00
2,agg_q07a_rain_yield_association.csv,26,0.00
3,agg_q10a_market_volatility_mh.csv,1058,0.07
4,agg_q11a_msp_gap_2026.csv,19,0.00
5,agg_q11b_msp_gap_by_month_2026.csv,60,0.00
6,agg_q11c_dispersion_2026.csv,18,0.00
7,agg_q11e_volatility_by_crop_2026.csv,14,0.00
8,agg_q11f_daily_median_price_2026.csv,312,0.01
9,agg_q11g_season_2026_outlook.csv,19,0.00


## Conclusion
The Power BI project is generated and validated against Microsoft's schemas.

## Limitations
Visuals need a Refresh in Power BI Desktop; screenshots are taken manually.

## Next steps
Publish to Power BI Service with scheduled refresh.